In [1]:
# ================================
# 04 - OPTİMİZASYON V2
# Kombinasyon tabanlı spot seçimi
# ================================

import pandas as pd
import numpy as np
from itertools import product

# Verileri yükle


tahmin       = pd.read_excel(r"C:\Users\semanur\Desktop\HB_Yarisma\data2\tahminlenen_talep.xlsx")
kiralik      = pd.read_excel(r"C:\Users\semanur\Desktop\HB_Yarisma\data\Kiralık_Araçlar.xlsx")
arac_maliyet = pd.read_excel(r"C:\Users\semanur\Desktop\HB_Yarisma\data\Araç_Kapasite_Maliyet.xlsx")
koordinat    = pd.read_excel(r"C:\Users\semanur\Desktop\HB_Yarisma\data\Koordinatlar v2.xlsx")
mesafe       = pd.read_csv(r"C:\Users\semanur\Desktop\HB_Yarisma\data2\mesafe_matrisi.csv", index_col=0)

# Rename
tahmin.columns       = ['tarih', 'cikis', 'varis', 'tahmin_desi']
kiralik.columns      = ['cikis', 'varis', 'arac_sayisi', 'arac_turu']
arac_maliyet.columns = ['arac_adi', 'kapasite', 'kiralik_kira',
                        'kiralik_km', 'spot_sabit', 'spot_km']

tahmin['tarih'] = pd.to_datetime(tahmin['tarih'])
arac_dict = arac_maliyet.set_index('arac_adi').to_dict('index')

print("✅ Veriler yüklendi")
print("\nAraç tipleri:")
for arac, bilgi in arac_dict.items():
    print(f"  {arac}: kapasite={bilgi['kapasite']}, "
          f"spot_sabit={bilgi['spot_sabit']}, spot_km={bilgi['spot_km']}")

✅ Veriler yüklendi

Araç tipleri:
  Tır: kapasite=22400, spot_sabit=11700, spot_km=25
  Kamyon: kapasite=12000, spot_sabit=7638, spot_km=21
  Hafif Kamyon: kapasite=7200, spot_sabit=8750, spot_km=20
  Kamyonet: kapasite=5600, spot_sabit=4750, spot_km=18


In [13]:
# ================================
# YARDIMCI FONKSİYONLAR
# ================================

def mesafe_al(cikis, varis):
    try:
        return float(mesafe.loc[cikis, varis])
    except:
        return 0

def spot_maliyet_hesapla(arac_adi, km):
    bilgi = arac_dict[arac_adi]
    return bilgi['spot_sabit'] + (km * bilgi['spot_km'])

def kiralik_maliyet_hesapla(arac_adi, km):
    bilgi = arac_dict[arac_adi]
    return bilgi['kiralik_kira'] + (km * bilgi['kiralik_km'])

def en_ucuz_spot_kombinasyon(kalan_desi, km):
    """
    Kalan desiyi taşıyacak en ucuz spot araç kombinasyonunu bul.
    
    Mantık:
    - Her araç tipinden 0,1,2,3... adet dene
    - Toplam kapasite >= kalan_desi olmalı
    - Her araç %10 doluluk şartını sağlamalı
    - En ucuz kombinasyonu döndür
    """
    arac_tipleri = ['Kamyonet', 'Hafif Kamyon', 'Kamyon', 'Tır']
    en_ucuz = None
    en_dusuk_maliyet = float('inf')

    # Her araç tipinden max kaç adet gerekebilir?
    # Kalan desi / o aracın kapasitesi + 1 (tavan)
    max_adet = {}
    for arac in arac_tipleri:
        kapasite = arac_dict[arac]['kapasite']
        max_adet[arac] = min(3, int(np.ceil(kalan_desi / kapasite)) + 1)

    # Tüm kombinasyonları dene
    for k, h, ka, t in product(
        range(max_adet['Kamyonet'] + 1),
        range(max_adet['Hafif Kamyon'] + 1),
        range(max_adet['Kamyon'] + 1),
        range(max_adet['Tır'] + 1)
    ):
        # Hiç araç yok, atla
        if k + h + ka + t == 0:
            continue

        # Toplam kapasite
        toplam_kapasite = (
            k  * arac_dict['Kamyonet']['kapasite'] +
            h  * arac_dict['Hafif Kamyon']['kapasite'] +
            ka * arac_dict['Kamyon']['kapasite'] +
            t  * arac_dict['Tır']['kapasite']
        )

        # Kapasite yetmiyor, atla
        if toplam_kapasite < kalan_desi:
            continue

        # %10 doluluk kontrolü - her araç için
        # Yükü büyükten küçüğe araçlara dağıt
        kalan = kalan_desi
        doluluk_ok = True
        arac_yuk = {}

        for arac, adet in [('Tır',t),('Kamyon',ka),
                           ('Hafif Kamyon',h),('Kamyonet',k)]:
            kapasite = arac_dict[arac]['kapasite']
            for i in range(adet):
                yuk = min(kapasite, kalan)
                if yuk / kapasite < 0.10:
                    doluluk_ok = False
                    break
                kalan -= yuk
                arac_yuk[arac] = arac_yuk.get(arac, []) + [yuk]
            if not doluluk_ok:
                break

        if not doluluk_ok:
            continue

        # Maliyet hesapla
        maliyet = (
            k  * spot_maliyet_hesapla('Kamyonet', km) +
            h  * spot_maliyet_hesapla('Hafif Kamyon', km) +
            ka * spot_maliyet_hesapla('Kamyon', km) +
            t  * spot_maliyet_hesapla('Tır', km)
        )

        if maliyet < en_dusuk_maliyet:
            en_dusuk_maliyet = maliyet
            en_ucuz = {
                'Kamyonet': k, 'Hafif Kamyon': h,
                'Kamyon': ka, 'Tır': t,
                'toplam_maliyet': maliyet,
                'arac_yuk': arac_yuk
            }

    return en_ucuz

# Test edelim
print("TEST: 8000 desi, İstanbul→Kocaeli (78 km)")
km = mesafe_al('İstanbul', 'Kocaeli')
sonuc = en_ucuz_spot_kombinasyon(8000, km)
print(f"En ucuz kombinasyon: {sonuc}")

TEST: 8000 desi, İstanbul→Kocaeli (78 km)
En ucuz kombinasyon: {'Kamyonet': 0, 'Hafif Kamyon': 0, 'Kamyon': 1, 'Tır': 0, 'toplam_maliyet': 9271.75430434866, 'arac_yuk': {'Kamyon': [8000]}}


In [14]:
# ================================
# ANA OPTİMİZASYON DÖNGÜSÜ V3
# Kiralık araçlar her gün zorunlu
# ================================

sonuclar = []

for _, row in tahmin.iterrows():
    tarih      = row['tarih']
    cikis      = row['cikis']
    varis      = row['varis']
    kalan_desi = row['tahmin_desi']
    km         = mesafe_al(cikis, varis)

    # Bu rotada kiralık araç var mı?
    kiralik_rota = kiralik[
        (kiralik['cikis'] == cikis) &
        (kiralik['varis'] == varis)
    ]

    # ---- ADIM 1: KİRALIK ARAÇLAR - HER GÜN ZORUNLU ----
    if len(kiralik_rota) > 0:
        arac_turu   = kiralik_rota.iloc[0]['arac_turu']
        arac_sayisi = kiralik_rota.iloc[0]['arac_sayisi']
        kapasite    = arac_dict[arac_turu]['kapasite']
        maliyet_birim = kiralik_maliyet_hesapla(arac_turu, km)

        # Her araç için - dolu olsun boş olsun plana gir
        for i in range(arac_sayisi):
            yuk = min(kapasite, kalan_desi)  # talep yoksa 0 olur
            yuk = max(yuk, 0)

            sonuclar.append({
                'Tarih'      : tarih,
                'Araç Tipi'  : f'Kiralık {arac_turu}',
                'Çıkış TM'   : cikis,
                'Varış TM'   : varis,
                'Atanan Desi': round(yuk, 2),
                'Maliyet'    : round(maliyet_birim, 2)
            })
            kalan_desi -= yuk
            kalan_desi  = max(kalan_desi, 0)

    # ---- ADIM 2: KALAN DESİ İÇİN SPOT (%10 DOLULUK ŞARTI) ----
    if kalan_desi > 0:
        kombinasyon = en_ucuz_spot_kombinasyon(kalan_desi, km)

        if kombinasyon:
            for arac_adi, adet in [('Tır', kombinasyon['Tır']),
                                    ('Kamyon', kombinasyon['Kamyon']),
                                    ('Hafif Kamyon', kombinasyon['Hafif Kamyon']),
                                    ('Kamyonet', kombinasyon['Kamyonet'])]:
                if adet > 0:
                    yukler = kombinasyon['arac_yuk'].get(arac_adi, [])
                    for yuk in yukler:
                        maliyet = spot_maliyet_hesapla(arac_adi, km)
                        sonuclar.append({
                            'Tarih'      : tarih,
                            'Araç Tipi'  : f'Spot {arac_adi}',
                            'Çıkış TM'   : cikis,
                            'Varış TM'   : varis,
                            'Atanan Desi': round(yuk, 2),
                            'Maliyet'    : round(maliyet, 2)
                        })

sonuc_df = pd.DataFrame(sonuclar)

print(f"Toplam araç atama : {len(sonuc_df)}")
print(f"Kiralık           : {len(sonuc_df[sonuc_df['Araç Tipi'].str.startswith('Kiralık')])}")
print(f"Spot              : {len(sonuc_df[sonuc_df['Araç Tipi'].str.startswith('Spot')])}")
print(f"\nTOPLAM MALİYET: {sonuc_df['Maliyet'].sum():,.0f} TL")

Toplam araç atama : 682
Kiralık           : 98
Spot              : 584

TOPLAM MALİYET: 10,061,463 TL


In [5]:
# Boş giden kiralık araçları göster
bos_kiralik = sonuc_df[
    (sonuc_df['Araç Tipi'].str.startswith('Kiralık')) &
    (sonuc_df['Atanan Desi'] == 0)
]

print(f"Boş giden kiralık araç sayısı: {len(bos_kiralik)}")
print(bos_kiralik.to_string())

Boş giden kiralık araç sayısı: 3
         Tarih    Araç Tipi  Çıkış TM   Varış TM  Atanan Desi  Maliyet
408 2026-05-17  Kiralık Tır  İstanbul  Eskişehir          0.0  9459.97
652 2026-05-14  Kiralık Tır  İstanbul     Yalova          0.0  7606.25
658 2026-05-17  Kiralık Tır  İstanbul     Yalova          0.0  7606.25


In [6]:
atanan = sonuc_df.groupby(
    ['Tarih','Çıkış TM','Varış TM']
)['Atanan Desi'].sum().reset_index()

kontrol = tahmin.merge(
    atanan,
    left_on=['tarih','cikis','varis'],
    right_on=['Tarih','Çıkış TM','Varış TM'],
    how='left'
)

kontrol['Atanan Desi'] = kontrol['Atanan Desi'].fillna(0)
kontrol['fark'] = kontrol['tahmin_desi'] - kontrol['Atanan Desi']

print("Maksimum fark:", kontrol['fark'].abs().max())
print("Hatalı satır:", (kontrol['fark'].abs() > 0.01).sum())

Maksimum fark: 551.3
Hatalı satır: 33


In [7]:
# Hangi rotalar eksik kalmış?
eksik = kontrol[kontrol['fark'].abs() > 0.01][
    ['tarih','cikis','varis','tahmin_desi','Atanan Desi','fark']
].sort_values('fark', ascending=False)

print("Eksik kalan rotalar:")
print(eksik.to_string())

Eksik kalan rotalar:
         tarih      cikis      varis  tahmin_desi  Atanan Desi    fark
531 2026-05-17     Yalova    Bilecik       551.30          0.0  551.30
405 2026-05-17  Eskişehir  Balıkesir       531.05          0.0  531.05
104 2026-05-17   Tekirdağ  Balıkesir       506.66          0.0  506.66
251 2026-05-17   Tekirdağ    Bilecik       461.27          0.0  461.27
76  2026-05-17   Tekirdağ  Şanlıurfa       445.58          0.0  445.58
258 2026-05-17  Eskişehir    Bilecik       415.60          0.0  415.60
552 2026-05-17     Yalova    Kütahya       390.72          0.0  390.72
272 2026-05-17  Balıkesir   İstanbul       369.55          0.0  369.55
433 2026-05-17   Tekirdağ    Kütahya       361.62          0.0  361.62
69  2026-05-17  Eskişehir     Mersin       357.66          0.0  357.66
111 2026-05-17   Tekirdağ    Karaman       338.87          0.0  338.87
454 2026-05-17  Eskişehir  Şanlıurfa       338.55          0.0  338.55
412 2026-05-17   Tekirdağ    Isparta       337.46       

In [10]:
sonuc_df.to_excel(
    r"C:\Users\semanur\Desktop\HB_Yarisma\data2\arac_planlamasi_v3.xlsx",
    index=False
)

print("✅ Kaydedildi!")
print(f"\n{'='*45}")
print(f"TOPLAM MALİYET : {sonuc_df['Maliyet'].sum():,.0f} TL")
print(f"Toplam araç    : {len(sonuc_df)}")
print(f"Kiralık        : {len(sonuc_df[sonuc_df['Araç Tipi'].str.startswith('Kiralık')])}")
print(f"Spot           : {len(sonuc_df[sonuc_df['Araç Tipi'].str.startswith('Spot')])}")
print(f"Taşınamayan    : 33 rota ← depoda kaldı, SLA cezası yok")
print(f"{'='*45}")

✅ Kaydedildi!

TOPLAM MALİYET : 10,061,463 TL
Toplam araç    : 682
Kiralık        : 98
Spot           : 584
Taşınamayan    : 33 rota ← depoda kaldı, SLA cezası yok


UĞRAMA EKLENİRSE

In [19]:
gun_istanbul = sonuc_df[
    (sonuc_df['Tarih'] == '2026-05-11') &
    (sonuc_df['Çıkış TM'] == 'İstanbul') &
    (sonuc_df['Araç Tipi'].str.startswith('Spot'))
]
print(gun_istanbul.to_string())

         Tarih      Araç Tipi  Çıkış TM   Varış TM  Atanan Desi   Maliyet
285 2026-05-11       Spot Tır  İstanbul    Isparta     15357.46  21332.26
298 2026-05-11       Spot Tır  İstanbul    Bilecik     22400.00  14885.60
299 2026-05-11  Spot Kamyonet  İstanbul    Bilecik      3462.58   7043.64
319 2026-05-11    Spot Kamyon  İstanbul  Balıkesir     12000.00  11364.51
320 2026-05-11    Spot Kamyon  İstanbul  Balıkesir     11013.25  11364.51
332 2026-05-11    Spot Kamyon  İstanbul     Manisa     12000.00  13870.45
333 2026-05-11    Spot Kamyon  İstanbul     Manisa     11501.50  13870.45
345 2026-05-11       Spot Tır  İstanbul     Mardin     20061.63  38984.20
352 2026-05-11       Spot Tır  İstanbul  Zonguldak     12928.46  17725.98
359 2026-05-11       Spot Tır  İstanbul    Denizli     13629.08  20686.71
366 2026-05-11       Spot Tır  İstanbul   Erzincan     21473.96  34239.38
373 2026-05-11       Spot Tır  İstanbul    Karaman     12157.46  25752.68
380 2026-05-11       Spot Tır  İstanbu

In [20]:
import itertools

istanbul_spot = sonuc_df[
    (sonuc_df['Tarih'] == '2026-05-11') &
    (sonuc_df['Çıkış TM'] == 'İstanbul') &
    (sonuc_df['Araç Tipi'].str.startswith('Spot'))
].copy()

print("Birleştirilebilecek çiftler (kapasite sığıyor):")
for (i1, r1), (i2, r2) in itertools.combinations(istanbul_spot.iterrows(), 2):
    toplam_yuk = r1['Atanan Desi'] + r2['Atanan Desi']
    
    # Hangi araç tipine sığar?
    for arac in ['Kamyonet','Hafif Kamyon','Kamyon','Tır']:
        kapasite = arac_dict[arac]['kapasite']
        if toplam_yuk <= kapasite:
            # Maliyet karşılaştır
            km1 = mesafe_al('İstanbul', r1['Varış TM'])
            km2 = mesafe_al('İstanbul', r2['Varış TM'])
            km_ugrama = km1 + mesafe_al(r1['Varış TM'], r2['Varış TM'])
            
            maliyet_ayri = r1['Maliyet'] + r2['Maliyet']
            maliyet_ugrama = spot_maliyet_hesapla(arac, km_ugrama)
            
            if maliyet_ugrama < maliyet_ayri:
                tasarruf = maliyet_ayri - maliyet_ugrama
                print(f"✅ {r1['Varış TM']} + {r2['Varış TM']} → "
                      f"{arac} ile uğramalı | "
                      f"Tasarruf: {tasarruf:,.0f} TL")
            break

Birleştirilebilecek çiftler (kapasite sığıyor):
✅ Isparta + Bilecik → Tır ile uğramalı | Tasarruf: 295 TL
✅ Isparta + Eskişehir → Tır ile uğramalı | Tasarruf: 2,563 TL
✅ Isparta + Mersin → Tır ile uğramalı | Tasarruf: 12,444 TL
✅ Bilecik + Zonguldak → Tır ile uğramalı | Tasarruf: 4,608 TL
✅ Bilecik + Denizli → Tır ile uğramalı | Tasarruf: 5,969 TL
✅ Bilecik + Karaman → Tır ile uğramalı | Tasarruf: 7,044 TL
✅ Bilecik + Sivas → Tır ile uğramalı | Tasarruf: 6,195 TL
✅ Bilecik + Eskişehir → Kamyon ile uğramalı | Tasarruf: 3,582 TL
✅ Bilecik + Mersin → Kamyon ile uğramalı | Tasarruf: 7,024 TL
✅ Bilecik + Kütahya → Tır ile uğramalı | Tasarruf: 6,729 TL
✅ Bilecik + Yalova → Kamyon ile uğramalı | Tasarruf: 3,625 TL
✅ Balıkesir + Mersin → Tır ile uğramalı | Tasarruf: 350 TL
✅ Balıkesir + Mersin → Tır ile uğramalı | Tasarruf: 350 TL
✅ Zonguldak + Eskişehir → Tır ile uğramalı | Tasarruf: 2,764 TL
✅ Zonguldak + Mersin → Tır ile uğramalı | Tasarruf: 7,537 TL
✅ Zonguldak + Yalova → Tır ile uğramalı 

In [21]:
# ================================
# UĞRAMA OPTİMİZASYONU
# Greedy Matching
# ================================

import itertools

def ugrama_optimize_et(gun_df, tarih):
    """
    Bir günün spot araçlarını uğrama ile optimize et.
    Her çıkış noktası için ayrı ayrı çalışır.
    """
    sonuc_satirlar = []
    kullanilan_indexler = set()
    
    # Her çıkış noktası için ayrı çalış
    for cikis in gun_df['Çıkış TM'].unique():
        cikis_df = gun_df[gun_df['Çıkış TM'] == cikis].copy()
        
        # Tasarruflu çiftleri bul
        tasarruflar = []
        for (i1, r1), (i2, r2) in itertools.combinations(cikis_df.iterrows(), 2):
            toplam_yuk = r1['Atanan Desi'] + r2['Atanan Desi']
            
            for arac in ['Kamyonet','Hafif Kamyon','Kamyon','Tır']:
                kapasite = arac_dict[arac]['kapasite']
                if toplam_yuk <= kapasite:
                    # İki yön dene: A→B→C ve A→C→B
                    km_bc = mesafe_al(r1['Varış TM'], r2['Varış TM'])
                    km_cb = mesafe_al(r2['Varış TM'], r1['Varış TM'])
                    
                    km1 = mesafe_al(cikis, r1['Varış TM'])
                    km2 = mesafe_al(cikis, r2['Varış TM'])
                    
                    # Hangi sıra daha kısa?
                    km_ugrama_1 = km1 + km_bc  # önce r1, sonra r2
                    km_ugrama_2 = km2 + km_cb  # önce r2, sonra r1
                    km_ugrama = min(km_ugrama_1, km_ugrama_2)
                    
                    maliyet_ayri   = r1['Maliyet'] + r2['Maliyet']
                    maliyet_ugrama = spot_maliyet_hesapla(arac, km_ugrama)
                    tasarruf = maliyet_ayri - maliyet_ugrama
                    
                    if tasarruf > 0:
                        tasarruflar.append({
                            'i1': i1, 'i2': i2,
                            'r1': r1, 'r2': r2,
                            'arac': arac,
                            'km_ugrama': km_ugrama,
                            'maliyet_ugrama': maliyet_ugrama,
                            'tasarruf': tasarruf
                        })
                    break
        
        # En yüksek tasarruftan başla - greedy matching
        tasarruflar.sort(key=lambda x: x['tasarruf'], reverse=True)
        
        for t in tasarruflar:
            i1, i2 = t['i1'], t['i2']
            
            # Bu rotalar daha önce kullanıldı mı?
            if i1 in kullanilan_indexler or i2 in kullanilan_indexler:
                continue
            
            # Uğramalı araç ekle
            r1, r2 = t['r1'], t['r2']
            sonuc_satirlar.append({
                'Tarih'      : tarih,
                'Araç Tipi'  : f"Spot {t['arac']} (Uğramalı)",
                'Çıkış TM'   : cikis,
                'Varış TM'   : f"{r1['Varış TM']} → {r2['Varış TM']}",
                'Atanan Desi': round(r1['Atanan Desi'] + r2['Atanan Desi'], 2),
                'Maliyet'    : round(t['maliyet_ugrama'], 2)
            })
            
            kullanilan_indexler.add(i1)
            kullanilan_indexler.add(i2)
        
        # Uğramaya girmeyen rotaları olduğu gibi ekle
        for i, r in cikis_df.iterrows():
            if i not in kullanilan_indexler:
                sonuc_satirlar.append(r.to_dict())
    
    return sonuc_satirlar, kullanilan_indexler

# Test: sadece 11 Mayıs spot araçlarında dene
gun_test = sonuc_df[
    (sonuc_df['Tarih'] == pd.Timestamp('2026-05-11')) &
    (sonuc_df['Araç Tipi'].str.startswith('Spot'))
].copy()

test_sonuc, kullanilan = ugrama_optimize_et(gun_test, pd.Timestamp('2026-05-11'))
test_df = pd.DataFrame(test_sonuc)

ugrama_sayisi = len(test_df[test_df['Araç Tipi'].str.contains('Uğramalı')])
print(f"Uğramalı araç sayısı   : {ugrama_sayisi}")
print(f"Orijinal araç sayısı   : {len(gun_test)}")
print(f"Yeni araç sayısı       : {len(test_df)}")
print(f"\nOrijinal maliyet : {gun_test['Maliyet'].sum():,.0f} TL")
print(f"Yeni maliyet     : {test_df['Maliyet'].sum():,.0f} TL")
print(f"Tasarruf         : {gun_test['Maliyet'].sum() - test_df['Maliyet'].sum():,.0f} TL")

Uğramalı araç sayısı   : 28
Orijinal araç sayısı   : 109
Yeni araç sayısı       : 81

Orijinal maliyet : 1,878,105 TL
Yeni maliyet     : 1,634,201 TL
Tasarruf         : 243,904 TL


In [23]:
# ================================
# TÜM HAFTAYA UĞRAMA UYGULA
# ================================

tum_sonuclar = []

for tarih in sonuc_df['Tarih'].unique():
    
    # O günün kiralık araçlarını olduğu gibi al
    kiralik_gun = sonuc_df[
        (sonuc_df['Tarih'] == tarih) &
        (sonuc_df['Araç Tipi'].str.startswith('Kiralık'))
    ]
    tum_sonuclar.extend(kiralik_gun.to_dict('records'))
    
    # O günün spot araçlarını uğrama ile optimize et
    spot_gun = sonuc_df[
        (sonuc_df['Tarih'] == tarih) &
        (sonuc_df['Araç Tipi'].str.startswith('Spot'))
    ].copy()
    
    if len(spot_gun) > 0:
        gun_sonuc, _ = ugrama_optimize_et(spot_gun, tarih)
        tum_sonuclar.extend(gun_sonuc)

final_df = pd.DataFrame(tum_sonuclar)

# Sonuçları karşılaştır
print(f"{'='*50}")
print(f"V3 (uğramasız) : {sonuc_df['Maliyet'].sum():,.0f} TL")
print(f"V4 (uğramalı)  : {final_df['Maliyet'].sum():,.0f} TL")
print(f"Toplam tasarruf: {sonuc_df['Maliyet'].sum() - final_df['Maliyet'].sum():,.0f} TL")
print(f"{'='*50}")
print(f"\nAraç sayısı:")
print(f"V3: {len(sonuc_df)}")
print(f"V4: {len(final_df)}")
print(f"\nGünlük tasarruf:")
for tarih in sorted(sonuc_df['Tarih'].unique()):
    v3 = sonuc_df[sonuc_df['Tarih']==tarih]['Maliyet'].sum()
    v4 = final_df[final_df['Tarih']==tarih]['Maliyet'].sum()
    print(f"{str(tarih)[:10]}: {v3-v4:,.0f} TL tasarruf")

V3 (uğramasız) : 10,061,463 TL
V4 (uğramalı)  : 8,313,828 TL
Toplam tasarruf: 1,747,635 TL

Araç sayısı:
V3: 682
V4: 472

Günlük tasarruf:
2026-05-11: 243,904 TL tasarruf
2026-05-12: 270,313 TL tasarruf
2026-05-13: 258,870 TL tasarruf
2026-05-14: 256,406 TL tasarruf
2026-05-15: 236,029 TL tasarruf
2026-05-16: 288,615 TL tasarruf
2026-05-17: 193,499 TL tasarruf


In [24]:
# Tekirdağ → Mardin uğrama örneği kontrol
km_trd = mesafe_al('İstanbul', 'Tekirdağ')
km_mrd = mesafe_al('İstanbul', 'Mardin')
km_ugrama = km_trd + mesafe_al('Tekirdağ', 'Mardin')

print(f"İstanbul → Tekirdağ      : {km_trd:.0f} km")
print(f"İstanbul → Mardin        : {km_mrd:.0f} km")
print(f"Tekirdağ → Mardin        : {mesafe_al('Tekirdağ','Mardin'):.0f} km")
print(f"Uğramalı toplam          : {km_ugrama:.0f} km")

# Maliyet karşılaştır (TIR)
m_ayri = spot_maliyet_hesapla('Tır', km_trd) + spot_maliyet_hesapla('Tır', km_mrd)
m_ugrama = spot_maliyet_hesapla('Tır', km_ugrama)

print(f"\n2 ayrı TIR maliyeti : {m_ayri:,.0f} TL")
print(f"1 uğramalı TIR      : {m_ugrama:,.0f} TL")
print(f"Tasarruf            : {m_ayri - m_ugrama:,.0f} TL")

İstanbul → Tekirdağ      : 123 km
İstanbul → Mardin        : 1091 km
Tekirdağ → Mardin        : 1208 km
Uğramalı toplam          : 1331 km

2 ayrı TIR maliyeti : 53,763 TL
1 uğramalı TIR      : 44,986 TL
Tasarruf            : 8,777 TL
